# Cache Aside 패턴과 Pipeline

Redis를 애플리케이션 캐시로 활용하는 **Cache Aside 패턴**과  
여러 명령을 묶어 처리하는 **Pipeline**을 배웁니다.

In [ ]:
import os
import json
import time
import redis
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    db=int(os.getenv('REDIS_DB', 0)),
    password=os.getenv('REDIS_PASSWORD') or None,
    decode_responses=True,
)

print(r.ping())  # True

## 1. Cache Aside 패턴

가장 일반적인 캐시 사용 패턴입니다.

```
요청 → Redis에서 읽기
  ├─ 캐시 히트(Hit) : Redis에서 바로 반환
  └─ 캐시 미스(Miss): DB 조회 → Redis에 저장 → 반환
```

| 상황 | 처리 | 속도 |
|---|---|---|
| 캐시 히트 | Redis에서 즉시 반환 | 빠름 |
| 캐시 미스 | DB 조회 후 Redis에 저장 | 느림 (최초 1회) |

In [ ]:
def db_query_product(product_id: int) -> dict:
    """실제 서비스에서는 RDB를 조회합니다. (0.1초 지연 시뮬레이션)"""
    time.sleep(0.1)
    return {'id': product_id, 'name': f'상품 {product_id}', 'price': product_id * 1000}


def get_product(product_id: int, ttl: int = 60) -> dict:
    cache_key = f'cache:product:{product_id}'
    cached = r.get(cache_key)

    if cached:
        print(f'[캐시 히트] {cache_key}')
        return json.loads(cached)

    print(f'[캐시 미스] {cache_key}')
    product = db_query_product(product_id)
    r.set(cache_key, json.dumps(product, ensure_ascii=False), ex=ttl)
    return product

In [ ]:
# 첫 번째 요청 — 캐시 미스 (DB 조회 발생)
r.delete('cache:product:42')

start = time.time()
product = get_product(42)
print(f'결과: {product}')
print(f'응답 시간: {(time.time() - start) * 1000:.1f}ms')

In [ ]:
# 두 번째 요청 — 캐시 히트 (Redis에서 즉시 반환)
start = time.time()
product = get_product(42)
print(f'결과: {product}')
print(f'응답 시간: {(time.time() - start) * 1000:.1f}ms')

In [ ]:
# 히트 / 미스 응답 시간 비교
r.delete('cache:product:99')

records = []
for i in range(5):
    start = time.time()
    get_product(99)
    elapsed_ms = (time.time() - start) * 1000
    status = '캐시 미스' if i == 0 else '캐시 히트'
    records.append({'요청 번호': i + 1, '상태': status, '응답시간(ms)': round(elapsed_ms, 2)})

pd.DataFrame(records)

## 2. Pipeline

여러 명령을 한 번에 묶어 서버로 보냅니다. 네트워크 왕복 횟수를 줄여 성능이 향상됩니다.

```
일반    : 명령1 → 응답1, 명령2 → 응답2, 명령3 → 응답3  (왕복 3번)
Pipeline: [명령1, 명령2, 명령3] → [응답1, 응답2, 응답3]  (왕복 1번)
```

> 순서가 보장되며 중간에 오류가 나도 나머지 명령은 실행됩니다.  
> 원자적 처리가 필요하면 `r.pipeline(transaction=True)`를 사용합니다.

In [ ]:
# 게시글 5개의 조회수를 Pipeline으로 일괄 조회
article_ids = [101, 102, 103, 104, 105]

# 초기값 설정
for aid in article_ids:
    r.set(f'views:article:{aid}', aid * 10)

# Pipeline으로 일괄 조회
pipe = r.pipeline()
for aid in article_ids:
    pipe.get(f'views:article:{aid}')
views = pipe.execute()

df = pd.DataFrame({'게시글': [f'article:{aid}' for aid in article_ids], '조회수': views})
df

In [ ]:
# 일반 방식 vs Pipeline 성능 비교
n = 200
keys = [f'bench:{i}' for i in range(n)]

# 일반 방식
start = time.time()
for k in keys:
    r.set(k, 1)
normal_ms = (time.time() - start) * 1000

# Pipeline 방식
start = time.time()
pipe = r.pipeline()
for k in keys:
    pipe.set(k, 1)
pipe.execute()
pipeline_ms = (time.time() - start) * 1000

# 정리
pipe = r.pipeline()
for k in keys:
    pipe.delete(k)
pipe.execute()

pd.DataFrame([
    {'방식': '일반', '명령 수': n, '소요시간(ms)': round(normal_ms, 1)},
    {'방식': 'Pipeline', '명령 수': n, '소요시간(ms)': round(pipeline_ms, 1)},
])

## 3. ConnectionPool

웹 서버처럼 동시 요청이 많은 환경에서는 매 요청마다 새 연결을 만들면 비용이 큽니다.  
**ConnectionPool**로 연결을 미리 만들어 재사용합니다.

| 방식 | 설명 |
|---|---|
| 연결을 매번 생성 | 연결 비용 발생, 동시 처리 어려움 |
| ConnectionPool | 연결 재사용, 동시 요청 안정적으로 처리 |

In [ ]:
pool = redis.ConnectionPool(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    db=int(os.getenv('REDIS_DB', 0)),
    decode_responses=True,
    max_connections=10,
)
r_pool = redis.Redis(connection_pool=pool)
print(r_pool.ping())  # True
print(f'최대 연결 수: {pool.max_connections}')

## 실습

1. 상품 3개(`product:1`, `product:2`, `product:3`)를 DB에서 조회하는 함수를 만들고,  
   Cache Aside 패턴을 적용하세요. 첫 번째와 두 번째 조회의 응답 시간을 비교하세요.
2. Pipeline으로 사용자 10명의 포인트(`points:user:1` ~ `points:user:10`)를 한 번에 저장하고,  
   결과를 DataFrame으로 출력하세요.
3. 위 포인트를 일반 방식과 Pipeline 방식으로 각각 읽어 소요 시간을 비교하세요.

### 1. 상품 3개에 Cache Aside 패턴 적용 및 응답 시간 비교

In [ ]:
# 1. 상품 3개에 Cache Aside 패턴 적용 및 응답 시간 비교
def db_query_ex(product_id: int) -> dict:
    time.sleep(0.1)
    return {'id': product_id, 'name': f'실습상품 {product_id}', 'price': product_id * 5000}

def get_product_ex(product_id: int, ttl: int = 60) -> dict:
    cache_key = f'ex:cache:product:{product_id}'
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), '캐시 히트'
    product = db_query_ex(product_id)
    r.set(cache_key, json.dumps(product, ensure_ascii=False), ex=ttl)
    return product, '캐시 미스'

# 캐시 초기화
for pid in [1, 2, 3]:
    r.delete(f'ex:cache:product:{pid}')

# 각 상품 2번씩 요청: 첫 번째는 미스, 두 번째는 히트
records = []
for pid in [1, 2, 3, 1, 2, 3]:
    start = time.time()
    _, status = get_product_ex(pid)
    elapsed_ms = (time.time() - start) * 1000
    records.append({'product_id': pid, '상태': status, '응답시간(ms)': round(elapsed_ms, 2)})

pd.DataFrame(records)

### 2. Pipeline으로 사용자 10명 포인트 일괄 저장 후 DataFrame 출력

In [ ]:
# 2. Pipeline으로 사용자 10명 포인트 일괄 저장 후 DataFrame 출력
pipe = r.pipeline()
for i in range(1, 11):
    pipe.set(f'points:user:{i}', i * 100)
pipe.execute()

# Pipeline으로 일괄 조회
pipe = r.pipeline()
for i in range(1, 11):
    pipe.get(f'points:user:{i}')
points = pipe.execute()

df = pd.DataFrame({
    '사용자': [f'user:{i}' for i in range(1, 11)],
    '포인트': [int(p) for p in points],
})
df

### 3. 일반 방식 vs Pipeline 읽기 시간 비교 (위에서 저장한 포인트 사용)

In [ ]:
# 3. 일반 방식 vs Pipeline 읽기 시간 비교 (위에서 저장한 포인트 사용)
# 일반 방식
start = time.time()
for i in range(1, 11):
    r.get(f'points:user:{i}')
normal_ms = (time.time() - start) * 1000

# Pipeline 방식
start = time.time()
pipe = r.pipeline()
for i in range(1, 11):
    pipe.get(f'points:user:{i}')
pipe.execute()
pipeline_ms = (time.time() - start) * 1000

# 키 정리
pipe = r.pipeline()
for i in range(1, 11):
    pipe.delete(f'points:user:{i}')
pipe.execute()

pd.DataFrame([
    {'방식': '일반',     '명령 수': 10, '소요시간(ms)': round(normal_ms, 2)},
    {'방식': 'Pipeline', '명령 수': 10, '소요시간(ms)': round(pipeline_ms, 2)},
])